# ISOM 835 · Session 7 — Decision Trees & Random Forests
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Nov 2 · Prof. Hasan Arslan**

How a tree asks questions, why it memorizes, and how averaging hundreds of decorrelated trees turns a fragile model into the workhorse of tabular ML.

> **Frame the prediction (Hotel Bookings).** *Unit:* one reservation · *Target:* `is_canceled` · *Horizon:* at booking · *Decision:* overbooking and deposit policy · *Base rate:* 37% canceled.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import roc_auc_score

URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/hotel_bookings.csv'
raw = pd.read_csv(URL)
print(raw.shape, f'cancellation rate {raw.is_canceled.mean():.1%}')
LEAKS = ['reservation_status', 'reservation_status_date', 'assigned_room_type']   # outcome in disguise / known only at check-in
df = raw.sample(30_000, random_state=835)                                          # subsample for speed in class
X = df.drop(columns=['is_canceled'] + LEAKS)
X = X.apply(lambda c: c.astype('category').cat.codes if c.dtype == object else c).fillna(-1)   # trees: ordinal codes are fine
y = df['is_canceled']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=835)

## 1. How a tree asks questions
Each node picks the split that most reduces **Gini impurity** — the probability that two random guests from the node disagree on cancellation. A depth-3 tree is the whole cancellation story on one screen.

In [ ]:
t3 = DecisionTreeClassifier(max_depth=3, random_state=835).fit(X_tr, y_tr)
plt.figure(figsize=(18, 6)); plot_tree(t3, feature_names=X.columns, class_names=['stay', 'cancel'], filled=True, impurity=True, fontsize=8, proportion=True); plt.show()
print(f'depth-3 tree test AUC {roc_auc_score(y_te, t3.predict_proba(X_te)[:, 1]):.3f}')

## 2. Watching a tree overfit
Train vs. test AUC as depth grows: the unlimited tree memorizes the training bookings.

In [ ]:
depths = [2, 4, 6, 8, 10, 14, 20, None]; tr, te = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=835).fit(X_tr, y_tr)
    tr.append(roc_auc_score(y_tr, t.predict_proba(X_tr)[:, 1])); te.append(roc_auc_score(y_te, t.predict_proba(X_te)[:, 1]))
lab = [str(d) for d in depths]
plt.figure(figsize=(7, 3.5)); plt.plot(lab, tr, 'o-', color='#2ee6c5', label='train'); plt.plot(lab, te, 'o-', color='#ff6b8b', label='test'); plt.xlabel('max_depth'); plt.ylabel('AUC'); plt.legend(); plt.title('validation curve over depth'); plt.show()
print(pd.DataFrame({'depth': lab, 'train': np.round(tr, 3), 'test': np.round(te, 3)}).to_string(index=False))

In [ ]:
# min_samples_leaf is often the better brake for business data
for leaf in [1, 5, 20, 50, 200]:
    t = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=835).fit(X_tr, y_tr)
    print(f'min_samples_leaf={leaf:3d}: leaves {t.get_n_leaves():5d}   test AUC {roc_auc_score(y_te, t.predict_proba(X_te)[:, 1]):.3f}')

## 3. From one tree to a forest
**Bagging**: many deep trees on bootstrap samples, averaged → variance drops. **Random forest**: bagging plus a random subset of features at every split → the trees disagree more, and the average improves more. **Out-of-bag score**: a free validation set from the rows each tree never saw.

In [ ]:
bag = BaggingClassifier(DecisionTreeClassifier(random_state=835), n_estimators=100, n_jobs=-1, random_state=835).fit(X_tr, y_tr)
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, oob_score=True, n_jobs=-1, random_state=835).fit(X_tr, y_tr)
print(f'single unlimited tree  test AUC {te[-1]:.3f}')
print(f'bagging (100 trees)    test AUC {roc_auc_score(y_te, bag.predict_proba(X_te)[:, 1]):.3f}')
print(f'random forest (300)    test AUC {roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1]):.3f}   OOB accuracy {rf.oob_score_:.3f}')

In [ ]:
# Forests are hard to overfit by adding trees
for n in [10, 50, 100, 300]:
    m = RandomForestClassifier(n_estimators=n, min_samples_leaf=5, n_jobs=-1, random_state=835).fit(X_tr, y_tr)
    print(f'{n:3d} trees: test AUC {roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]):.3f}')

## 4. Which features matter — two answers that disagree
`feature_importances_` (impurity) rewards columns with many split points, whether or not they predict. **Permutation importance** on held-out data asks the honest question: *if I scrambled this column, how much worse would the model get?* We add a pure-noise column to catch the difference.

In [ ]:
Xn_tr = X_tr.assign(noise=np.random.default_rng(835).normal(size=len(X_tr))); Xn_te = X_te.assign(noise=np.random.default_rng(1).normal(size=len(X_te)))
rf2 = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=835).fit(Xn_tr, y_tr)
imp = pd.DataFrame({'impurity': rf2.feature_importances_}, index=Xn_tr.columns)
perm = permutation_importance(rf2, Xn_te, y_te, scoring='roc_auc', n_repeats=5, random_state=835, n_jobs=-1)
imp['permutation'] = perm.importances_mean
print(imp.sort_values('permutation', ascending=False).head(10).round(4))
print(f"\nnoise column rank: impurity #{int((imp['impurity'] > imp.loc['noise', 'impurity']).sum()) + 1}   permutation #{int((imp['permutation'] > imp.loc['noise', 'permutation']).sum()) + 1}   of {len(imp)}")

## 5. Partial dependence — the *shape* of an effect
Importance says *how much*; partial dependence says *which way and how*. Cancellation probability vs. lead time rises steeply and then flattens — a curve a revenue manager can price.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
PartialDependenceDisplay.from_estimator(rf, X_te.sample(3000, random_state=835), ['lead_time', 'previous_cancellations', 'total_of_special_requests'], ax=ax, kind='average')
plt.tight_layout(); plt.show()

## 6. Your turn
1. **Leak test.** Put `assigned_room_type` back in. How much does the forest's AUC rise — and would that signal exist at booking time?
2. **Depth vs. leaf.** Which single-tree brake gives the higher test AUC on this data, `max_depth` or `min_samples_leaf`? Report the best setting for each.
3. **Competition transfer.** The course file *is* the competition's training set (83,573 rows; the 35,817 test rows exist only on Kaggle). Train the forest on all of it — drop the 30k subsample — score the Kaggle `test.csv`, write a `sample_submission`-shaped file, and submit tonight. How does the public leaderboard compare with your local test AUC?

In [ ]:
# Your turn — work here

## What we learned tonight
- A decision tree is a **flowchart learned greedily**: interpretable, scale-free, high-variance. Depth and `min_samples_leaf` are its brakes.
- A random forest is **variance reduction by averaging decorrelated trees**; hard to overfit by adding trees; the OOB score is free validation.
- **Impurity importance lies about high-cardinality columns; permutation importance on held-out data does not.** Partial dependence shows the shape.

Competition closes Sun Nov 8. Project proposals due Mon Nov 9. Read ISLP 8.2.3 (Boosting).